In [1]:
import numpy as np

In [2]:
class PCAScratch:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None

    def fit(self, X):
        # 1. Centratura dei dati (Mean Centering)
        self.mean = np.mean(X, axis=0)
        X = X - self.mean

        # 2. Calcolo della Matrice di Covarianza
        # rowvar=False perché le nostre feature sono nelle colonne
        covariance_matrix = np.cov(X, rowvar=False)

        # 3. Calcolo di Autovettori e Autovalori (Eigenvectors & Eigenvalues)
        eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

        # 4. Ordinamento
        # Gli autovettori sono restituiti in ordine crescente, li invertiamo
        eigenvectors = eigenvectors.T
        idxs = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idxs]
        eigenvectors = eigenvectors[idxs]

        # 5. Selezione delle prime 'n' componenti
        self.components = eigenvectors[0:self.n_components]

    def transform(self, X):
        # Proiezione dei dati sui nuovi assi
        X = X - self.mean
        return np.dot(X, self.components.T)

In [4]:
# --- Test con dati giocattolo ---
# Immaginiamo dati 2D che vogliamo ridurre a 1D
X = np.array([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6]]) 

pca = PCAScratch(n_components=1)
pca.fit(X)
X_projected = pca.transform(X)

print("Dati Originali (2D):\n", X)
print("\nDati Proiettati (1D):\n", X_projected)

Dati Originali (2D):
 [[1 2]
 [2 3]
 [3 4]
 [4 5]
 [5 6]]

Dati Proiettati (1D):
 [[-2.82842712]
 [-1.41421356]
 [ 0.        ]
 [ 1.41421356]
 [ 2.82842712]]


In [5]:
import torch

In [6]:
class PCAPyTorch:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None

    def fit(self, X):
        # 1. Media e Centratura
        # X deve essere un tensore float
        self.mean = torch.mean(X, dim=0)
        X_centered = X - self.mean

        # 2. Singular Value Decomposition (SVD)
        # U: matrici unitarie, S: valori singolari, V: autovettori (V.T è quello che ci serve)
        # full_matrices=False è più efficiente
        U, S, V = torch.linalg.svd(X_centered, full_matrices=False)

        # 3. Selezione delle prime 'n' componenti
        # In SVD, V contiene già gli autovettori ordinati per importanza
        self.components = V[:self.n_components]

    def transform(self, X):
        # Proiezione: (X - media) * V^T
        X_centered = X - self.mean
        return torch.mm(X_centered, self.components.t())

In [11]:
# --- Test con i tuoi dati ---
X_np = [[1.0, 2.0], [2.0, 3.0], [3.0, 4.0], [4.0, 5.0], [5.0, 6.0]]
X = torch.tensor(X_np)

pca = PCAPyTorch(n_components=1)
pca.fit(X)
X_projected = pca.transform(X)

print("Dati Originali (Tensor):\n", X)
print("\nDati Proiettati (Tensor):\n", X_projected)

Dati Originali (Tensor):
 tensor([[1., 2.],
        [2., 3.],
        [3., 4.],
        [4., 5.],
        [5., 6.]])

Dati Proiettati (Tensor):
 tensor([[-2.8284],
        [-1.4142],
        [ 0.0000],
        [ 1.4142],
        [ 2.8284]])
